In [9]:
import pandas as pd
import numpy as np
import json
import ast
import warnings
import traceback
from datetime import datetime
from pathlib import Path
from functools import lru_cache

warnings.filterwarnings("ignore", category=Warning)
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

RUN_LOG_PATH = None


def set_run_log(path=None, reset=False):
    global RUN_LOG_PATH
    RUN_LOG_PATH = str(path) if path else None
    if RUN_LOG_PATH and reset:
        log_file = Path(RUN_LOG_PATH)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        log_file.write_text("", encoding="utf-8")


def write_run_log(message):
    if not RUN_LOG_PATH:
        return
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_file = Path(RUN_LOG_PATH)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with log_file.open("a", encoding="utf-8") as handle:
        handle.write(f"[{timestamp}] {message}\n")


def progress_iter(iterable, desc="", total=None, enabled=True):
    if not enabled:
        return iterable

    if total is None:
        try:
            total = len(iterable)
        except TypeError:
            total = None

    wrapped_iterable = tqdm(iterable, desc=desc, total=total) if tqdm is not None else iterable

    def _generator():
        if total in (None, 0):
            if desc:
                log_stage(desc, enabled=True)
            for item in wrapped_iterable:
                yield item
            return

        step = max(1, total // 10)
        for idx, item in enumerate(wrapped_iterable, 1):
            if idx == 1 or idx == total or idx % step == 0:
                log_stage(f"{desc}: {idx}/{total}", enabled=True)
            yield item

    return _generator()


def log_stage(message, enabled=True):
    if enabled:
        print(message, flush=True)
    write_run_log(message)


def safe_to_number(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        if pd.isna(value):
            return None
        return int(value) if isinstance(value, float) and value.is_integer() else value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            number = float(text)
        except Exception:
            return None
        return int(number) if number.is_integer() else number
    return None


@lru_cache(maxsize=50000)
def _parse_json_text(text):
    text = text.strip()
    if not text or text == "{}":
        return {}
    try:
        obj = json.loads(text)
    except Exception:
        try:
            obj = ast.literal_eval(text)
        except Exception:
            return {}
    if not isinstance(obj, dict):
        return {}

    normalized = {}
    for key, value in obj.items():
        number = safe_to_number(value)
        normalized[str(key)] = number if number is not None else value
    return normalized


def parse_json_like(value):
    if value is None:
        return {}
    if isinstance(value, float) and pd.isna(value):
        return {}
    if isinstance(value, dict):
        normalized = {}
        for key, raw_value in value.items():
            number = safe_to_number(raw_value)
            normalized[str(key)] = number if number is not None else raw_value
        return normalized
    if isinstance(value, str):
        return dict(_parse_json_text(value))
    return {}


def merge_max_dicts(dicts):
    """默认规则：同 key 取最大值；非数值场景保留后一个值。"""
    merged = {}
    for item in dicts:
        current = parse_json_like(item)
        for key, value in current.items():
            current_number = safe_to_number(value)
            previous_number = safe_to_number(merged.get(key))
            if current_number is not None:
                if previous_number is None or current_number > previous_number:
                    merged[key] = current_number
            else:
                merged[key] = value
    return merged


def merge_sum_dicts(dicts):
    """fail_out 专用：同 key 直接累加；非数值场景保留后一个值。"""
    merged = {}
    for item in dicts:
        current = parse_json_like(item)
        for key, value in current.items():
            current_number = safe_to_number(value)
            previous_number = safe_to_number(merged.get(key))
            if current_number is not None:
                merged[key] = (previous_number or 0) + current_number
            else:
                merged[key] = value
    return merged


def dict_max(value):
    current = parse_json_like(value)
    return max(current.values()) if current else 0


def new_org(front, back):
    diff_counts = []
    diff_dicts = []
    for left, right in zip(front, back):
        left_dict = parse_json_like(left)
        right_dict = parse_json_like(right)
        diff_dict = {key: value for key, value in left_dict.items() if key not in right_dict}
        diff_counts.append(len(diff_dict))
        diff_dicts.append(diff_dict)
    return (
        pd.Series(diff_counts, index=front.index),
        pd.Series(diff_dicts, index=front.index, dtype="object"),
    )


def safe_divide(numerator, denominator):
    result = numerator.div(denominator.replace(0, np.nan))
    return result.replace([np.inf, -np.inf], np.nan)


def first_non_zero(frame):
    values = frame.to_numpy()
    mask = values != 0
    result = np.full(values.shape[0], 99999, dtype=int)
    has_non_zero = mask.any(axis=1)
    if has_non_zero.any():
        result[has_non_zero] = mask[has_non_zero].argmax(axis=1)
    return pd.Series(result, index=frame.index)


def last_non_zero(frame):
    values = frame.to_numpy()
    mask = values != 0
    result = np.full(values.shape[0], 99999, dtype=int)
    has_non_zero = mask.any(axis=1)
    if has_non_zero.any():
        result[has_non_zero] = values.shape[1] - 1 - mask[has_non_zero, ::-1].argmax(axis=1)
    return pd.Series(result, index=frame.index)


def build_activity_frame(cnt_frame, amt_frame):
    cnt_positive = cnt_frame.fillna(0).gt(0) if cnt_frame.shape[1] else pd.DataFrame(False, index=cnt_frame.index, columns=cnt_frame.columns)
    amt_positive = amt_frame.fillna(0).gt(0) if amt_frame.shape[1] else pd.DataFrame(False, index=amt_frame.index, columns=amt_frame.columns)
    values = cnt_positive.to_numpy(dtype=bool) | amt_positive.to_numpy(dtype=bool)
    columns = list(cnt_frame.columns) if cnt_frame.shape[1] else list(amt_frame.columns)
    return pd.DataFrame(values.astype(int), index=cnt_frame.index if cnt_frame.shape[0] else amt_frame.index, columns=columns)


def merge_dict_columns(frame, merge_func=merge_max_dicts):
    if frame.shape[1] == 0:
        return pd.Series([{} for _ in range(len(frame))], index=frame.index, dtype="object")
    merged = [merge_func(items) for items in frame.itertuples(index=False, name=None)]
    return pd.Series(merged, index=frame.index, dtype="object")

In [ ]:
input_path = ""
output_name = "文件名"
target_vars_path = "/Users/ellingtonchen/Desktop/TC/支付共建/百创指数三期字典515.csv"
strict_target_vars = False  # 若需要发现非 verify 目标变量缺失时直接报错，可改为 True。
key_cols = ("idcard", "dateback")  # 按输入数据真实列名填写，例如 ("idcard", "dateBack")。
run_log_path = "v7_run.log"  # 运行日志会持续写入该文件；断连后可直接打开查看进度和错误。
build_checkpoint_dir = "v7_build_df_new_ckpt"  # build_df_new 断点目录；设为 None 可关闭断点续跑。
resume_build_checkpoint = True  # 断线/失败后重跑时复用断点；若要从头重跑，改为 False 或删除断点目录。
set_run_log(run_log_path, reset=True)
log_stage(f"V7 运行日志初始化: {run_log_path}")

# 原始流程：CSV 先读入 pandas；Parquet 路径交给 build_df_new 做低内存 lazy 聚合。
def read_input_file(path):
    if not path:
        return None
    path_text = str(path)
    lower_path = path_text.lower()
    if lower_path.endswith(".parquet"):
        return path_text
    if lower_path.endswith((".csv", ".csv.gz")):
        return pd.read_csv(path_text)
    raise ValueError(f"不支持的输入文件格式: {path_text}，请使用 .csv 或 .parquet")


df = read_input_file(input_path)

In [7]:
# 保留原始主流程：先用 build_df_new 做预聚合，再用 get 生成结果。

In [ ]:
# 下面按 build_df_new -> get 的顺序执行。

In [ ]:
def build_df_new(
    df,
    key_cols=("idcard", "dateback"),
    card_col="card",
    serialize_max=True,
    show_progress=True,
    use_polars_numeric=True,
    numeric_batch_size=128,
    max_batch_size=8,
    checkpoint_dir=None,
    resume_checkpoint=True,
):
    """
    低内存预聚合：
    1. Parquet 输入可直接传文件路径，数值列用 Polars lazy 分批 groupby，避免先全量读入 pandas
    2. *_max 字典列继续用 pandas/Python 解析，但按小批列读取和聚合，避免构造超宽 object 中间表
    3. pandas DataFrame 输入仍可运行；此时不再复制整表，只在分批处理时复制必要列
    4. checkpoint_dir 不为空时会保存 df_new 断点，重跑时跳过已完成的批次
    """
    import gc
    from pathlib import Path

    key_cols = list(key_cols)
    id_col = key_cols[0]
    source_path = str(df) if isinstance(df, (str, Path)) else None
    df_src = None if source_path else df
    pl = None
    checkpoint_root = Path(checkpoint_dir) if checkpoint_dir else None
    checkpoint_state_path = checkpoint_root / "df_new.pkl" if checkpoint_root else None
    checkpoint_meta_path = checkpoint_root / "metadata.json" if checkpoint_root else None
    checkpoint_metadata = {}

    def collect_polars(lazy_frame):
        try:
            return lazy_frame.collect(streaming=True)
        except TypeError:
            return lazy_frame.collect()

    def chunked(values, size):
        size = max(1, int(size))
        for start in range(0, len(values), size):
            yield values[start:start + size]

    def normalize_merge_keys(frame):
        # Polars 与 pandas 读取 parquet 时可能把同一 key 推成不同 dtype；merge 前统一为字符串。
        for key in key_cols:
            if key in frame.columns:
                frame[key] = frame[key].astype("string").fillna("<NA>")
        if id_col in frame.columns:
            frame[id_col] = frame[id_col].astype("string").fillna("<NA>")
        return frame

    def save_checkpoint(frame, stage):
        if checkpoint_root is None:
            return
        checkpoint_root.mkdir(parents=True, exist_ok=True)
        tmp_state_path = checkpoint_state_path.with_suffix(".tmp.pkl")
        frame.to_pickle(tmp_state_path)
        tmp_state_path.replace(checkpoint_state_path)
        metadata = dict(checkpoint_metadata)
        metadata["last_stage"] = stage
        metadata["updated_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        tmp_meta_path = checkpoint_meta_path.with_suffix(".tmp.json")
        tmp_meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        tmp_meta_path.replace(checkpoint_meta_path)
        log_stage(f"build_df_new checkpoint 已保存: {stage} -> {checkpoint_state_path}", show_progress)

    def load_checkpoint_if_valid(expected_metadata):
        if checkpoint_root is None or not resume_checkpoint:
            return None
        if not checkpoint_state_path.exists() or not checkpoint_meta_path.exists():
            return None
        saved_metadata = json.loads(checkpoint_meta_path.read_text(encoding="utf-8"))
        compare_keys = ["source_path", "key_cols", "card_col", "all_cols", "sum_cols", "max_cols"]
        mismatched_keys = [key for key in compare_keys if saved_metadata.get(key) != expected_metadata.get(key)]
        if mismatched_keys:
            raise ValueError(
                f"发现 checkpoint 但 metadata 不匹配: {mismatched_keys}。"
                f"请删除 {checkpoint_root}，或更换 checkpoint_dir，或设置 resume_build_checkpoint=False 后从头运行。"
            )
        log_stage(f"从 build_df_new checkpoint 恢复: {checkpoint_state_path}，上次阶段: {saved_metadata.get('last_stage')}", show_progress)
        return normalize_merge_keys(pd.read_pickle(checkpoint_state_path))

    if source_path:
        if not source_path.lower().endswith(".parquet"):
            raise ValueError(f"build_df_new 仅支持 DataFrame 或 parquet 路径输入: {source_path}")
        if use_polars_numeric:
            try:
                import polars as pl
            except ImportError as error:
                raise ImportError("Parquet 低内存预聚合需要安装 polars；或传入 pandas DataFrame / use_polars_numeric=False。") from error
            scan = pl.scan_parquet(source_path)
            try:
                all_cols = list(scan.collect_schema().names())
            except Exception:
                all_cols = list(scan.schema.keys())
        else:
            df_src = pd.read_parquet(source_path)
            source_path = None
            all_cols = list(df_src.columns)
    else:
        all_cols = list(df_src.columns)

    missing_keys = [col for col in key_cols if col not in all_cols]
    if missing_keys:
        raise KeyError(f"主键列不存在: {missing_keys}")

    if card_col not in all_cols and "cardnum" not in all_cols:
        raise KeyError(f"卡号列不存在: {card_col}，且也没有可复用的 cardnum 列")

    log_stage("开始 build_df_new 预聚合...", show_progress)
    if source_path:
        log_stage("检测到 parquet 路径输入：数值列使用 Polars lazy 聚合，*_max 列使用 pandas 分批聚合。", show_progress)
    else:
        log_stage("检测到 pandas DataFrame 输入：跳过整表 copy，按批处理必要列。", show_progress)

    max_cols = [col for col in all_cols if col.endswith("_max")]
    fail_out_max_cols = [col for col in max_cols if col.endswith("_fail_out_max")]
    fail_out_cnt_cols = [col for col in all_cols if col.endswith("_fail_out_cnt")]
    other_sum_cols = [
        col
        for col in all_cols
        if (col.endswith("_amt") or col.endswith("_cnt"))
        and not col.endswith("_fail_out_amt")
        and not col.endswith("_fail_out_cnt")
    ]
    sum_cols = fail_out_cnt_cols + other_sum_cols
    checkpoint_metadata = {
        "source_path": str(Path(source_path).resolve()) if source_path else None,
        "key_cols": key_cols,
        "card_col": card_col,
        "all_cols": all_cols,
        "sum_cols": sum_cols,
        "max_cols": max_cols,
    }
    if checkpoint_root is not None:
        log_stage(f"build_df_new checkpoint_dir: {checkpoint_root}，resume={resume_checkpoint}", show_progress)
        if not resume_checkpoint:
            for old_path in (checkpoint_state_path, checkpoint_meta_path):
                if old_path.exists():
                    old_path.unlink()

    def read_pandas_columns(columns):
        columns = [col for col in dict.fromkeys(columns) if col in all_cols]
        if source_path:
            return pd.read_parquet(source_path, columns=columns)
        return df_src[columns].copy()

    df_new = load_checkpoint_if_valid(checkpoint_metadata)
    if df_new is None:
        if source_path and use_polars_numeric:
            df_new = collect_polars(pl.scan_parquet(source_path).select(key_cols).unique(maintain_order=True)).to_pandas()
        else:
            df_new = df_src[key_cols].drop_duplicates().reset_index(drop=True)
        df_new = normalize_merge_keys(df_new)
        save_checkpoint(df_new, "init_keys")

    if sum_cols:
        sum_batches = list(chunked(sum_cols, numeric_batch_size))
        for batch_idx, cols in enumerate(progress_iter(sum_batches, desc="Polars/Pandas 分批聚合数值列", total=len(sum_batches), enabled=show_progress), 1):
            if all(col in df_new.columns for col in cols):
                log_stage(f"跳过已完成数值列批次 {batch_idx}/{len(sum_batches)}", show_progress)
                continue
            if source_path and use_polars_numeric:
                exprs = [
                    pl.col(col).cast(pl.Float64, strict=False).fill_null(0).sum().alias(col)
                    for col in cols
                ]
                part = collect_polars(
                    pl.scan_parquet(source_path)
                    .select(key_cols + cols)
                    .group_by(key_cols)
                    .agg(exprs)
                ).to_pandas()
            else:
                numeric_df = df_src[key_cols + cols].copy()
                numeric_df = normalize_merge_keys(numeric_df)
                numeric_df[cols] = numeric_df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)
                part = numeric_df.groupby(key_cols, dropna=False, as_index=False)[cols].sum()
                del numeric_df
            part = normalize_merge_keys(part)
            new_cols = [col for col in part.columns if col not in key_cols and col not in df_new.columns]
            df_new = df_new.merge(part[key_cols + new_cols], on=key_cols, how="left")
            save_checkpoint(df_new, f"numeric_batch_{batch_idx}_of_{len(sum_batches)}")
            del part
            gc.collect()

    if max_cols:
        max_batches = list(chunked(max_cols, max_batch_size))
        for batch_idx, cols in enumerate(progress_iter(max_batches, desc="pandas 分批解析并聚合 *_max", total=len(max_batches), enabled=show_progress), 1):
            if all(col in df_new.columns for col in cols):
                log_stage(f"跳过已完成 *_max 批次 {batch_idx}/{len(max_batches)}", show_progress)
                continue
            pending_cols = [col for col in cols if col not in df_new.columns]
            max_df = read_pandas_columns(key_cols + pending_cols)
            max_df = normalize_merge_keys(max_df)
            for col in pending_cols:
                max_df[col] = [parse_json_like(value) for value in max_df[col]]
            agg_map = {
                col: merge_sum_dicts if col.endswith("_fail_out_max") else merge_max_dicts
                for col in pending_cols
            }
            part = max_df.groupby(key_cols, dropna=False).agg(agg_map).reset_index()
            part = normalize_merge_keys(part)
            df_new = df_new.merge(part, on=key_cols, how="left")
            save_checkpoint(df_new, f"max_batch_{batch_idx}_of_{len(max_batches)}")
            del max_df, part
            gc.collect()

    generated_fail_amt_cols = []
    fail_out_amt_changed = False
    for col in progress_iter(
        fail_out_max_cols,
        desc="生成 fail_out 聚合字段",
        total=len(fail_out_max_cols),
        enabled=show_progress,
    ):
        if col not in df_new.columns:
            continue
        base_name = col[:-4]
        amt_col = f"{base_name}_amt"
        if amt_col in df_new.columns:
            generated_fail_amt_cols.append(amt_col)
            continue
        parsed_values = [parse_json_like(value) for value in df_new[col]]
        df_new[amt_col] = [
            sum((safe_to_number(item) or 0) for item in current.values())
            for current in parsed_values
        ]
        generated_fail_amt_cols.append(amt_col)
        fail_out_amt_changed = True
    if fail_out_amt_changed:
        save_checkpoint(df_new, "fail_out_amt_complete")
    else:
        log_stage("跳过已完成 fail_out_amt 聚合字段", show_progress)

    if "cardnum" in df_new.columns:
        log_stage("跳过已完成 cardnum 聚合", show_progress)
    else:
        if source_path and use_polars_numeric:
            if card_col in all_cols:
                cardnum_df = collect_polars(
                    pl.scan_parquet(source_path)
                    .select([id_col, card_col])
                    .group_by(id_col)
                    .agg(pl.col(card_col).drop_nulls().n_unique().alias("cardnum"))
                ).to_pandas()
            else:
                cardnum_df = collect_polars(
                    pl.scan_parquet(source_path)
                    .select([id_col, "cardnum"])
                    .group_by(id_col)
                    .agg(pl.col("cardnum").max().alias("cardnum"))
                ).to_pandas()
        else:
            if card_col in all_cols:
                cardnum_df = (
                    df_src.groupby(id_col, dropna=False)[card_col]
                    .nunique(dropna=True)
                    .reset_index(name="cardnum")
                )
            else:
                cardnum_df = (
                    df_src[[id_col, "cardnum"]]
                    .groupby(id_col, dropna=False, as_index=False)["cardnum"]
                    .max()
                )
        cardnum_df = normalize_merge_keys(cardnum_df)
        df_new = df_new.merge(cardnum_df, on=id_col, how="left")
        del cardnum_df
        save_checkpoint(df_new, "cardnum")
        gc.collect()

    if serialize_max:
        for col in progress_iter(max_cols, desc="序列化 *_max", total=len(max_cols), enabled=show_progress):
            if col in df_new.columns:
                df_new[col] = df_new[col].map(lambda value: json.dumps(parse_json_like(value), ensure_ascii=False, separators=(",", ":")))

    ordered_cols = key_cols + ["cardnum"] + max_cols + generated_fail_amt_cols + fail_out_cnt_cols + other_sum_cols
    ordered_cols = [col for col in dict.fromkeys(ordered_cols) if col in df_new.columns]
    remain_cols = [col for col in df_new.columns if col not in ordered_cols]
    df_new = df_new[ordered_cols + remain_cols]
    df_new["index"] = df_new.groupby(key_cols, sort=False).ngroup()
    save_checkpoint(df_new, "complete")

    log_stage(f"build_df_new 完成，共 {len(df_new)} 行。", show_progress)
    return df_new

In [ ]:
def get(
    df,
    file_name,
    show_progress=True,
    pre_aggregated=None,
    target_vars_path=None,
    target_vars=None,
    strict_target_vars=False,
    ignore_var_prefixes=("verify",),
    key_cols=("idcard", "dateback"),
):
    """
    V7 口径：
    - 延续当前 V6 的 build_df_new -> get 主流程、missing 处理规则和新增 fail_out 月份特征
    - 输出阶段按目标字典只保留指定变量；默认忽略 verify* 开头变量
    - 若对应窗口完全无流水，则变量记为 missing；若窗口内存在流水，则非比值型变量补 0
    - 所有比值型变量若分母为 0，则记为 missing
    - late_* / early_* 单独处理：只要对应目标流水不存在，就记为 missing
    """

    def normalize_target_name(value):
        if pd.isna(value):
            return None
        name = str(value).strip()
        return name or None

    def load_target_vars_from_csv(path):
        if not path:
            return None
        last_error = None
        for encoding in ("utf-8-sig", "utf-8", "gbk"):
            try:
                target_df = pd.read_csv(path, encoding=encoding)
                break
            except UnicodeDecodeError as error:
                last_error = error
        else:
            raise last_error
        candidate_cols = ["英文", "var_name", "变量名", "变量英文", "变量英文名", "字段名", "feature", "name"]
        name_col = next((col for col in candidate_cols if col in target_df.columns), target_df.columns[0])
        return [normalize_target_name(value) for value in target_df[name_col]]

    def resolve_target_vars():
        raw_vars = []
        if target_vars is not None:
            raw_vars.extend(target_vars)
        loaded_vars = load_target_vars_from_csv(target_vars_path)
        if loaded_vars is not None:
            raw_vars.extend(loaded_vars)
        if not raw_vars:
            return None

        prefixes = tuple(str(prefix).lower() for prefix in (ignore_var_prefixes or ()))
        result = []
        seen = set()
        for raw_value in raw_vars:
            name = normalize_target_name(raw_value)
            if name is None:
                continue
            if prefixes and name.lower().startswith(prefixes):
                continue
            if name not in seen:
                seen.add(name)
                result.append(name)
        return result

    key_cols = list(key_cols)
    id_col = key_cols[0]
    work_df = df.copy()

    required_cols = key_cols
    missing_required = [col for col in required_cols if col not in work_df.columns]
    if missing_required:
        raise KeyError(f"缺少必要字段: {missing_required}")

    if pre_aggregated is None:
        pre_aggregated = {"index", "cardnum"}.issubset(work_df.columns) and work_df["index"].is_unique

    if not pre_aggregated:
        log_stage("检测到输入未完全聚合，先执行 build_df_new ...", show_progress)
        work_df = build_df_new(
            work_df,
            key_cols=key_cols,
            card_col="card",
            serialize_max=False,
            show_progress=show_progress,
        )
    else:
        log_stage("检测到输入已聚合，跳过 build_df_new。", show_progress)

    cols_cnt = [col for col in work_df.columns if col.endswith("_cnt")]
    cols_amt = [col for col in work_df.columns if col.endswith("_amt")]
    cols_max = [col for col in work_df.columns if col.endswith("_max")]

    if cols_max:
        for col in progress_iter(cols_max, desc="准备 *_max 字段", total=len(cols_max), enabled=show_progress):
            work_df[col] = [parse_json_like(value) for value in work_df[col]]

    if work_df["index"].is_unique:
        keep_cols = ["index"] + key_cols + cols_cnt + cols_amt + cols_max
        data_all = work_df[keep_cols].copy()
    else:
        log_stage("按 index 聚合重复记录...", show_progress)
        grouped = work_df.groupby("index", sort=False, dropna=False)
        pieces = []
        numeric_cols = cols_cnt + cols_amt
        if numeric_cols:
            pieces.append(grouped[numeric_cols].sum())
        if cols_max:
            max_block = {}
            for col in progress_iter(cols_max, desc="聚合 index 层 *_max", total=len(cols_max), enabled=show_progress):
                max_block[col] = grouped[col].agg(merge_max_dicts)
            pieces.append(pd.DataFrame(max_block))
        agg_df = pd.concat(pieces, axis=1).reset_index() if pieces else work_df[["index"]].drop_duplicates().copy()
        key_df = work_df[["index"] + key_cols].drop_duplicates(subset=["index"])
        data_all = key_df.merge(agg_df, on="index", how="left")

    cardnum_df = (
        work_df[[id_col, "cardnum"]]
        .groupby(id_col, dropna=False, as_index=False)["cardnum"]
        .max()
    )
    data = data_all.merge(cardnum_df, on=id_col, how="left")

    date_legacy = [0, 1, 2, 3, 6, 12, 18, 23]
    date_current = [0, 1, 2, 3, 6, 12, 18, 24]
    org_type1 = ["bank", "cf", "top", "others", "nonloan"]
    org_type2 = ["bank", "cf", "top", "others"]
    org_type3 = ["bank", "cf", "top", "others", "all"]
    org_type4 = ["bank", "cf", "top", "others", "all", "nonloan"]
    state_add = [
        "suc_out_cnt",
        "suc_out_amt",
        "suc_out_max",
        "fail_out_cnt",
        "fail_out_amt",
        "fail_out_max",
        "suc_in_cnt",
        "suc_in_amt",
    ]

    data_merge_cache = {}
    monthly_all_cache = {}
    target_activity_cache = {}
    target_amt_activity_cache = {}
    global_window_activity_cache = {}

    def merge_data_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in data_merge_cache:
            data_merge_cache[key] = merge_dict_columns(data[columns], merge_func=merge_func)
        return data_merge_cache[key]

    def dict_len_series(series):
        return pd.Series([len(parse_json_like(value)) for value in series], index=series.index)

    def dict_max_series(series):
        return pd.Series([dict_max(value) for value in series], index=series.index)

    def get_window_months(window, include_current=False):
        if window == 0:
            return [0]
        if include_current:
            if window == 24:
                return list(range(24))
            return list(range(window + 1))
        if window == 1:
            return [1]
        return list(range(1, window + 1))

    def build_target_activity_frame(metric_name, org):
        key = (metric_name, org)
        if key not in target_activity_cache:
            if org == "all":
                orgs = org_type1 if metric_name == "suc_out" else org_type2
            else:
                orgs = [org]
            monthly_block = {}
            for month in range(24):
                cnt_cols = [
                    f"pre_{month}_{target_org}_{metric_name}_cnt"
                    for target_org in orgs
                    if f"pre_{month}_{target_org}_{metric_name}_cnt" in data.columns
                ]
                amt_cols = [
                    f"pre_{month}_{target_org}_{metric_name}_amt"
                    for target_org in orgs
                    if f"pre_{month}_{target_org}_{metric_name}_amt" in data.columns
                ]
                cnt_positive = data[cnt_cols].fillna(0).gt(0).any(axis=1) if cnt_cols else pd.Series(False, index=data.index)
                amt_positive = data[amt_cols].fillna(0).gt(0).any(axis=1) if amt_cols else pd.Series(False, index=data.index)
                monthly_block[f"pre_{month}_active"] = (cnt_positive | amt_positive).astype(int)
            target_activity_cache[key] = pd.DataFrame(monthly_block, index=data.index)
        return target_activity_cache[key]

    def build_target_amt_activity_frame(metric_name, org):
        key = (metric_name, org)
        if key not in target_amt_activity_cache:
            if org == "all":
                orgs = org_type1 if metric_name == "suc_out" else org_type2
            else:
                orgs = [org]
            monthly_block = {}
            for month in range(24):
                amt_cols = [
                    f"pre_{month}_{target_org}_{metric_name}_amt"
                    for target_org in orgs
                    if f"pre_{month}_{target_org}_{metric_name}_amt" in data.columns
                ]
                amt_positive = data[amt_cols].fillna(0).gt(0).any(axis=1) if amt_cols else pd.Series(False, index=data.index)
                monthly_block[f"pre_{month}_active"] = amt_positive.astype(int)
            target_amt_activity_cache[key] = pd.DataFrame(monthly_block, index=data.index)
        return target_amt_activity_cache[key]

    def get_window_activity_mask(months):
        key = tuple(months)
        if key not in global_window_activity_cache:
            metric_masks = []
            for metric_name in ["suc_in", "suc_out", "fail_out"]:
                activity_df = build_target_activity_frame(metric_name, "all")
                cols = [f"pre_{month}_active" for month in months]
                metric_masks.append(activity_df[cols].gt(0).any(axis=1))
            global_window_activity_cache[key] = metric_masks[0] | metric_masks[1] | metric_masks[2]
        return global_window_activity_cache[key]

    def is_ratio_column(col):
        return col.endswith(("avg", "amtprop", "cntprop", "ognprop", "newognprop", "inc", "yoy"))

    def get_column_window_masks(col):
        if col.startswith("rec_"):
            parts = col.split("_")
            if len(parts) > 1 and parts[1].isdigit():
                window = int(parts[1])
                if col.endswith("inc"):
                    compare_months = list(range(13, 24)) if window == 12 else list(range(window + 1, 2 * window + 1))
                    return [
                        get_window_activity_mask(get_window_months(window, include_current=False)),
                        get_window_activity_mask(compare_months),
                    ]
                if col.endswith("yoy"):
                    compare_months = list(range(13, 13 + window))
                    return [
                        get_window_activity_mask(get_window_months(window, include_current=False)),
                        get_window_activity_mask(compare_months),
                    ]
                return [get_window_activity_mask(get_window_months(window, include_current=True))]
        if col.startswith("r"):
            parts = col.split("_")
            if len(parts) > 1 and parts[0].startswith("r") and parts[0][1:].isdigit() and parts[1].isdigit():
                start_month = int(parts[0][1:])
                end_month = int(parts[1])
                return [
                    get_window_activity_mask(get_window_months(start_month, include_current=True)),
                    get_window_activity_mask(get_window_months(end_month, include_current=True)),
                ]
            if parts and parts[0].startswith("r") and parts[0][1:].isdigit():
                window = int(parts[0][1:])
                return [get_window_activity_mask(get_window_months(window, include_current=True))]
        return None

    def apply_window_activity_rules(frame):
        result = frame.copy()
        for col in result.columns:
            if col.startswith("late_") or col.startswith("early_"):
                continue
            masks = get_column_window_masks(col)
            if masks is None:
                continue
            valid_mask = masks[0].copy()
            for mask in masks[1:]:
                valid_mask = valid_mask & mask
            result[col] = result[col].where(valid_mask, np.nan)
            if not is_ratio_column(col) and pd.api.types.is_numeric_dtype(result[col]):
                result[col] = result[col].fillna(0).where(valid_mask, np.nan)
        return result

    def apply_time_activity_rules(frame):
        result = frame.copy()
        for col in [target_col for target_col in result.columns if target_col.startswith("late_") or target_col.startswith("early_")]:
            parts = col.split("_")
            if len(parts) < 3:
                continue
            org = parts[1]
            metric_name = "_".join(parts[2:])
            activity_df = build_target_activity_frame(metric_name, org)
            valid_mask = activity_df.gt(0).any(axis=1)
            result[col] = result[col].where(valid_mask, np.nan)
        return result

    # 该临时按月 all max 不依赖 rec_n 口径，本次统一复用。
    monthly_all_block = {}
    for month in progress_iter(range(24), desc="按月汇总 max", total=24, enabled=show_progress):
        for metric in ["suc_out_max", "fail_out_max"]:
            source_cols = [f"pre_{month}_{org}_{metric}" for org in org_type2]
            monthly_all_block[f"rec_{month}_{metric}_all"] = merge_data_cols(source_cols)
    monthly_all_max_df = pd.DataFrame(monthly_all_block, index=data.index)

    def merge_monthly_all_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in monthly_all_cache:
            monthly_all_cache[key] = merge_dict_columns(monthly_all_max_df[columns], merge_func=merge_func)
        return monthly_all_cache[key]

    def build_rec_base(date_points, include_current=False, desc_suffix=""):
        dt_local = pd.DataFrame(index=data.index)
        local_cache = {}

        def merge_local_cols(columns, merge_func=merge_max_dicts):
            key = (merge_func.__name__, tuple(columns))
            if key not in local_cache:
                local_cache[key] = merge_dict_columns(dt_local[columns], merge_func=merge_func)
            return local_cache[key]

        block = {}
        for i in progress_iter(date_points, desc=f"构建近X月指标{desc_suffix}", total=len(date_points), enabled=show_progress):
            months = get_window_months(i, include_current=include_current)
            for metric in state_add:
                orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
                for org in orgs:
                    target_col = f"rec_{i}_{org}_{metric}"
                    source_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                    if metric.endswith("max"):
                        block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else merge_data_cols(source_cols)
                    else:
                        block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else data[source_cols].sum(axis=1)
        dt_local = pd.concat([dt_local, pd.DataFrame(block, index=data.index)], axis=1)

        block = {}
        for i in progress_iter(date_points, desc=f"构建全机构汇总{desc_suffix}", total=len(date_points), enabled=show_progress):
            for metric in state_add:
                orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
                source_cols = [f"rec_{i}_{org}_{metric}" for org in orgs]
                target_col = f"rec_{i}_all_{metric}"
                if metric.endswith("max"):
                    block[target_col] = merge_local_cols(source_cols)
                else:
                    block[target_col] = dt_local[source_cols].sum(axis=1)
        dt_local = pd.concat([dt_local, pd.DataFrame(block, index=data.index)], axis=1)
        return dt_local

    def build_avg_prop(dt_source, date_points, desc_suffix=""):
        block = {}
        for i in progress_iter(date_points, desc=f"生成均值和占比{desc_suffix}", total=len(date_points), enabled=show_progress):
            for org in org_type3:
                block[f"rec_{i}_{org}_suc_out_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_suc_out_amt"], dt_source[f"rec_{i}_{org}_suc_out_cnt"])
                block[f"rec_{i}_{org}_fail_out_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_amt"], dt_source[f"rec_{i}_{org}_fail_out_cnt"])
                block[f"rec_{i}_{org}_suc_in_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_suc_in_amt"], dt_source[f"rec_{i}_{org}_suc_in_cnt"])

                total_amt = dt_source[f"rec_{i}_{org}_suc_out_amt"] + dt_source[f"rec_{i}_{org}_fail_out_amt"]
                total_cnt = dt_source[f"rec_{i}_{org}_suc_out_cnt"] + dt_source[f"rec_{i}_{org}_fail_out_cnt"]
                block[f"rec_{i}_{org}_fail_out_amtprop"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_amt"], total_amt)
                block[f"rec_{i}_{org}_fail_out_cntprop"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_cnt"], total_cnt)

            block[f"rec_{i}_nonloan_suc_out_avg"] = safe_divide(
                dt_source[f"rec_{i}_nonloan_suc_out_amt"],
                dt_source[f"rec_{i}_nonloan_suc_out_cnt"],
            )
        return pd.DataFrame(block, index=data.index)

    def build_org_features(dt_source, date_points, desc_suffix=""):
        future_merge_cache = {}

        def future_dict_series(current_month, org, status):
            key = (current_month, org, status)
            if key not in future_merge_cache:
                future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in range(current_month + 1, 24)]
                if future_cols:
                    future_merge_cache[key] = merge_data_cols(future_cols)
                else:
                    future_merge_cache[key] = pd.Series([{} for _ in range(len(data))], index=data.index, dtype="object")
            return future_merge_cache[key]

        block = {}
        max_window = max(date_points)
        for i in progress_iter(date_points, desc=f"生成机构数与新增机构{desc_suffix}", total=len(date_points), enabled=show_progress):
            for status in ["suc", "fail"]:
                for org in org_type3:
                    current_series = dt_source[f"rec_{i}_{org}_{status}_out_max"]
                    block[f"rec_{i}_{org}_{status}_out_ogn"] = dict_len_series(current_series)

                    if org != "all" and i != max_window:
                        future_series = future_dict_series(i, org, status)
                        new_cnt, new_max = new_org(current_series, future_series)
                        block[f"rec_{i}_{org}_{status}_out_newogn"] = new_cnt
                        block[f"rec_{i}_{org}_{status}_out_newognmax"] = new_max

                if i != max_window:
                    count_cols = [block[f"rec_{i}_{org}_{status}_out_newogn"] for org in org_type2]
                    block[f"rec_{i}_all_{status}_out_newogn"] = count_cols[0] + count_cols[1] + count_cols[2] + count_cols[3]
                    temp_df = pd.DataFrame(
                        {
                            org: block[f"rec_{i}_{org}_{status}_out_newognmax"]
                            for org in org_type2
                        },
                        index=data.index,
                    )
                    block[f"rec_{i}_all_{status}_out_newognmax"] = merge_dict_columns(temp_df)
        return pd.DataFrame(block, index=data.index)

    def build_time_compare(dt_source, compare_points, desc_suffix=""):
        dm_pairs = [(compare_points[left], compare_points[right]) for left in range(len(compare_points) - 1) for right in range(left + 1, len(compare_points))]
        max_compare = max(compare_points)
        block = {}
        for start_month, end_month in progress_iter(dm_pairs, desc=f"生成时间窗口对比{desc_suffix}", total=len(dm_pairs), enabled=show_progress):
            for org in org_type3:
                block[f"r{start_month}_{end_month}_{org}_suc_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_out_cnt"], dt_source[f"rec_{end_month}_{org}_suc_out_cnt"])
                block[f"r{start_month}_{end_month}_{org}_suc_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_out_amt"], dt_source[f"rec_{end_month}_{org}_suc_out_amt"])
                block[f"r{start_month}_{end_month}_{org}_fail_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_fail_out_cnt"], dt_source[f"rec_{end_month}_{org}_fail_out_cnt"])
                block[f"r{start_month}_{end_month}_{org}_fail_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_fail_out_amt"], dt_source[f"rec_{end_month}_{org}_fail_out_amt"])
                block[f"r{start_month}_{end_month}_{org}_suc_in_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_in_cnt"], dt_source[f"rec_{end_month}_{org}_suc_in_cnt"])
                block[f"r{start_month}_{end_month}_{org}_suc_in_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_in_amt"], dt_source[f"rec_{end_month}_{org}_suc_in_amt"])

                block[f"r{start_month}_{end_month}_{org}_suc_out_ad"] = dt_source[f"rec_{end_month}_{org}_suc_out_amt"] - dt_source[f"rec_{start_month}_{org}_suc_out_amt"]
                block[f"r{start_month}_{end_month}_{org}_fail_out_ad"] = dt_source[f"rec_{end_month}_{org}_fail_out_amt"] - dt_source[f"rec_{start_month}_{org}_fail_out_amt"]
                block[f"r{start_month}_{end_month}_{org}_suc_in_ad"] = dt_source[f"rec_{end_month}_{org}_suc_in_amt"] - dt_source[f"rec_{start_month}_{org}_suc_in_amt"]

                for status in ["suc", "fail"]:
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_ognprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_{status}_out_ogn"], dt_source[f"rec_{end_month}_{org}_{status}_out_ogn"])
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_od"] = dt_source[f"rec_{end_month}_{org}_{status}_out_ogn"] - dt_source[f"rec_{start_month}_{org}_{status}_out_ogn"]
                    if start_month != 12 and end_month != max_compare:
                        block[f"r{start_month}_{end_month}_{org}_{status}_out_newognprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_{status}_out_newogn"], dt_source[f"rec_{end_month}_{org}_{status}_out_newogn"])
                        block[f"r{start_month}_{end_month}_{org}_{status}_out_nod"] = dt_source[f"rec_{end_month}_{org}_{status}_out_newogn"] - dt_source[f"rec_{start_month}_{org}_{status}_out_newogn"]

            block[f"r{start_month}_{end_month}_nonloan_suc_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_nonloan_suc_out_cnt"], dt_source[f"rec_{end_month}_nonloan_suc_out_cnt"])
            block[f"r{start_month}_{end_month}_nonloan_suc_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_nonloan_suc_out_amt"], dt_source[f"rec_{end_month}_nonloan_suc_out_amt"])
            block[f"r{start_month}_{end_month}_nonloan_suc_out_ad"] = dt_source[f"rec_{end_month}_nonloan_suc_out_amt"] - dt_source[f"rec_{start_month}_nonloan_suc_out_amt"]
        return pd.DataFrame(block, index=data.index)

    def build_diff_metrics(dt_source, date_points, desc_suffix=""):
        block = {}
        for i in progress_iter(date_points, desc=f"生成差额指标{desc_suffix}", total=len(date_points), enabled=show_progress):
            for org in org_type3:
                block[f"r{i}_{org}_sf_out_cd"] = dt_source[f"rec_{i}_{org}_suc_out_cnt"] - dt_source[f"rec_{i}_{org}_fail_out_cnt"]
                block[f"r{i}_{org}_sf_out_ad"] = dt_source[f"rec_{i}_{org}_suc_out_amt"] - dt_source[f"rec_{i}_{org}_fail_out_amt"]
                block[f"r{i}_{org}_sf_out_mad"] = new_org(
                    dt_source[f"rec_{i}_{org}_suc_out_max"],
                    dt_source[f"rec_{i}_{org}_fail_out_max"],
                )[1]
                block[f"r{i}_{org}_suc_oi_cd"] = dt_source[f"rec_{i}_{org}_suc_out_cnt"] - dt_source[f"rec_{i}_{org}_suc_in_cnt"]
                block[f"r{i}_{org}_suc_oi_ad"] = dt_source[f"rec_{i}_{org}_suc_out_amt"] - dt_source[f"rec_{i}_{org}_suc_in_amt"]
        return pd.DataFrame(block, index=data.index)

    def build_time_features():
        block = {}
        for org in progress_iter(org_type2, desc="定位交易时间", total=len(org_type2), enabled=show_progress):
            suc_out_cnt_cols = [f"pre_{month}_{org}_suc_out_cnt" for month in range(24)]
            suc_out_amt_cols = [f"pre_{month}_{org}_suc_out_amt" for month in range(24)]
            fail_out_cnt_cols = [f"pre_{month}_{org}_fail_out_cnt" for month in range(24)]
            fail_out_amt_cols = [f"pre_{month}_{org}_fail_out_amt" for month in range(24)]
            suc_in_cnt_cols = [f"pre_{month}_{org}_suc_in_cnt" for month in range(24)]
            suc_in_amt_cols = [f"pre_{month}_{org}_suc_in_amt" for month in range(24)]

            suc_out_active = build_activity_frame(data[suc_out_cnt_cols], data[suc_out_amt_cols])
            fail_out_active = build_activity_frame(data[fail_out_cnt_cols], data[fail_out_amt_cols])
            suc_in_active = build_activity_frame(data[suc_in_cnt_cols], data[suc_in_amt_cols])

            block[f"late_{org}_suc_out"] = first_non_zero(suc_out_active)
            block[f"late_{org}_fail_out"] = first_non_zero(fail_out_active)
            block[f"late_{org}_suc_in"] = first_non_zero(suc_in_active)

            block[f"early_{org}_suc_out"] = last_non_zero(suc_out_active)
            block[f"early_{org}_fail_out"] = last_non_zero(fail_out_active)
            block[f"early_{org}_suc_in"] = last_non_zero(suc_in_active)

        nonloan_cnt_cols = [f"pre_{month}_nonloan_suc_out_cnt" for month in range(24)]
        nonloan_amt_cols = [f"pre_{month}_nonloan_suc_out_amt" for month in range(24)]
        nonloan_active = build_activity_frame(data[nonloan_cnt_cols], data[nonloan_amt_cols])
        block["late_nonloan_suc_out"] = first_non_zero(nonloan_active)
        block["early_nonloan_suc_out"] = last_non_zero(nonloan_active)

        temp_time = pd.DataFrame(block, index=data.index)
        temp_time["late_all_suc_out"] = temp_time[["late_bank_suc_out", "late_cf_suc_out", "late_top_suc_out", "late_others_suc_out", "late_nonloan_suc_out"]].min(axis=1)
        temp_time["late_all_fail_out"] = temp_time[["late_bank_fail_out", "late_cf_fail_out", "late_top_fail_out", "late_others_fail_out"]].min(axis=1)
        temp_time["late_all_suc_in"] = temp_time[["late_bank_suc_in", "late_cf_suc_in", "late_top_suc_in", "late_others_suc_in"]].min(axis=1)

        temp_time["early_all_suc_out"] = temp_time[["early_bank_suc_out", "early_cf_suc_out", "early_top_suc_out", "early_others_suc_out", "early_nonloan_suc_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        temp_time["early_all_fail_out"] = temp_time[["early_bank_fail_out", "early_cf_fail_out", "early_top_fail_out", "early_others_fail_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        temp_time["early_all_suc_in"] = temp_time[["early_bank_suc_in", "early_cf_suc_in", "early_top_suc_in", "early_others_suc_in"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        return temp_time

    def build_basic_inc(dt_source, desc_suffix=""):
        dm_inc = [1, 3, 6, 12]
        block = {}
        for i in progress_iter(dm_inc, desc=f"生成基础变量环比{desc_suffix}", total=len(dm_inc), enabled=show_progress):
            for metric in state_add:
                if metric in ["suc_out_cnt", "suc_out_amt"]:
                    target_orgs = org_type4
                elif metric.endswith("max"):
                    target_orgs = org_type4[:-1]
                else:
                    target_orgs = org_type4[:-1]

                for org in target_orgs:
                    source_name = f"rec_{i}_{org}_{metric}"
                    end_month = 23 if i == 12 else 2 * i

                    if metric.endswith("max"):
                        left = dict_max_series(dt_source[source_name])
                        if org != "all":
                            months = list(range(i + 1, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                            future_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                            right = dict_max_series(merge_data_cols(future_cols))
                        else:
                            months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                            future_cols = [f"rec_{month}_{metric}_all" for month in months]
                            right = dict_max_series(merge_monthly_all_cols(future_cols))
                        block[f"{source_name}inc"] = safe_divide(left, right)
                    else:
                        future_name = f"rec_{end_month}_{org}_{metric}"
                        diff_series = dt_source[future_name] - dt_source[source_name]
                        block[f"{source_name}inc"] = safe_divide(dt_source[source_name], diff_series)
        return pd.DataFrame(block, index=data.index)

    def build_org_inc(dt_source, desc_suffix=""):
        dm_inc = [1, 3, 6, 12]
        block = {}
        for i in progress_iter(dm_inc, desc=f"生成机构环比{desc_suffix}", total=len(dm_inc), enabled=show_progress):
            for org in org_type4[:-1]:
                for status in ["suc", "fail"]:
                    source_name = f"rec_{i}_{org}_{status}_out_ogn"
                    if org != "all":
                        months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in months]
                        right_series = dict_len_series(merge_data_cols(future_cols))
                    else:
                        months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"rec_{month}_{status}_out_max_all" for month in months]
                        right_series = dict_len_series(merge_monthly_all_cols(future_cols))
                    block[f"{source_name}inc"] = safe_divide(dt_source[source_name], right_series)
        return pd.DataFrame(block, index=data.index)

    def build_basic_yoy(dt_source, desc_suffix=""):
        dm_yoy = [1, 3, 6]
        block = {}
        for i in progress_iter(dm_yoy, desc=f"生成基础变量同比{desc_suffix}", total=len(dm_yoy), enabled=show_progress):
            compare_months = list(range(13, 13 + i))
            for metric in state_add:
                if metric in ["suc_out_cnt", "suc_out_amt"]:
                    target_orgs = org_type4
                    all_orgs = org_type1
                elif metric.endswith("max"):
                    target_orgs = org_type4[:-1]
                    all_orgs = org_type2
                else:
                    target_orgs = org_type4[:-1]
                    all_orgs = org_type2

                for org in target_orgs:
                    source_name = f"rec_{i}_{org}_{metric}"
                    target_name = f"{source_name}yoy"
                    if metric.endswith("max"):
                        left = dict_max_series(dt_source[source_name])
                        if org == "all":
                            compare_cols = [f"rec_{month}_{metric}_all" for month in compare_months]
                            right = dict_max_series(merge_monthly_all_cols(compare_cols))
                        else:
                            compare_cols = [f"pre_{month}_{org}_{metric}" for month in compare_months]
                            right = dict_max_series(merge_data_cols(compare_cols))
                        block[target_name] = safe_divide(left, right)
                    else:
                        if org == "all":
                            compare_cols = [
                                f"pre_{month}_{sub_org}_{metric}"
                                for month in compare_months
                                for sub_org in all_orgs
                            ]
                        else:
                            compare_cols = [f"pre_{month}_{org}_{metric}" for month in compare_months]
                        right = data[compare_cols].sum(axis=1)
                        block[target_name] = safe_divide(dt_source[source_name], right)
        return pd.DataFrame(block, index=data.index)

    def build_org_yoy(dt_source, desc_suffix=""):
        dm_yoy = [1, 3, 6]
        block = {}
        for i in progress_iter(dm_yoy, desc=f"生成机构同比{desc_suffix}", total=len(dm_yoy), enabled=show_progress):
            compare_months = list(range(13, 13 + i))
            for org in org_type4[:-1]:
                for status in ["suc", "fail"]:
                    source_name = f"rec_{i}_{org}_{status}_out_ogn"
                    target_name = f"{source_name}yoy"
                    if org != "all":
                        compare_cols = [f"pre_{month}_{org}_{status}_out_max" for month in compare_months]
                        right_series = dict_len_series(merge_data_cols(compare_cols))
                    else:
                        compare_cols = [f"rec_{month}_{status}_out_max_all" for month in compare_months]
                        right_series = dict_len_series(merge_monthly_all_cols(compare_cols))
                    block[target_name] = safe_divide(dt_source[source_name], right_series)
        return pd.DataFrame(block, index=data.index)

    def build_bcto(dt_source, cross_months, desc_suffix=""):
        block = {}
        for month in progress_iter(cross_months, desc=f"生成业务交叉指标{desc_suffix}", total=len(cross_months), enabled=show_progress):
            temp_cross = pd.DataFrame(index=data.index)
            for org in org_type2:
                cnt_cols = [
                    f"rec_{month}_{org}_suc_out_cnt",
                    f"rec_{month}_{org}_fail_out_cnt",
                    f"rec_{month}_{org}_suc_in_cnt",
                ]
                amt_cols = [
                    f"rec_{month}_{org}_suc_out_amt",
                    f"rec_{month}_{org}_fail_out_amt",
                    f"rec_{month}_{org}_suc_in_amt",
                ]
                has_cnt = dt_source[cnt_cols].fillna(0).gt(0).any(axis=1)
                has_amt = dt_source[amt_cols].fillna(0).gt(0).any(axis=1)
                temp_cross[org] = (has_cnt | has_amt).astype(int)
            block[f"r{month}_bcto"] = temp_cross[org_type2].sum(axis=1).ge(2).astype(float)
        return pd.DataFrame(block, index=data.index)

    def max_consecutive_positive(frame):
        values = frame.to_numpy() > 0
        current = np.zeros(values.shape[0], dtype=int)
        best = np.zeros(values.shape[0], dtype=int)
        for col_idx in range(values.shape[1]):
            current = (current + 1) * values[:, col_idx]
            best = np.maximum(best, current)
        return pd.Series(best, index=frame.index)

    def build_month_activity_features(desc_suffix=""):
        window_map = {
            6: list(range(7)),
            12: list(range(13)),
            24: list(range(24)),
        }
        block = {}
        window_items = list(window_map.items())
        for window, months in progress_iter(window_items, desc=f"生成月份活跃特征{desc_suffix}", total=len(window_items), enabled=show_progress):
            for metric_name in ["suc_in", "suc_out", "fail_out"]:
                activity_df = build_target_activity_frame(metric_name, "all")
                window_cols = [f"pre_{month}_active" for month in months]
                window_df = activity_df[window_cols]
                block[f"rec_{window}_all_{metric_name}_month_cnt"] = window_df.gt(0).sum(axis=1).astype(float)
                block[f"rec_{window}_all_{metric_name}_max_cont_month_cnt"] = max_consecutive_positive(window_df).astype(float)
        return pd.DataFrame(block, index=data.index)

    def build_fail_out_month_features(desc_suffix=""):
        target_windows = [3, 6, 12, 24]
        block = {}
        for window in progress_iter(target_windows, desc=f"生成失败转出月份特征{desc_suffix}", total=len(target_windows), enabled=show_progress):
            months = get_window_months(window, include_current=True)
            window_cols = [f"pre_{month}_active" for month in months]
            for org in org_type3:
                activity_df = build_target_amt_activity_frame("fail_out", org)
                window_df = activity_df[window_cols]
                block[f"rec_{window}_{org}_fail_out_mth"] = window_df.gt(0).sum(axis=1).clip(upper=window).astype(float)
                block[f"rec_{window}_{org}_fail_out_contimth"] = max_consecutive_positive(window_df).clip(upper=window).astype(float)
        return pd.DataFrame(block, index=data.index)

    log_stage("构建旧口径 rec_n（供第7/8层和同比使用）...", show_progress)
    legacy_base = build_rec_base(date_legacy, include_current=False, desc_suffix="（旧口径）")
    legacy_org = build_org_features(legacy_base, date_legacy, desc_suffix="（旧口径）")
    legacy_dt = pd.concat([legacy_base, legacy_org], axis=1)

    basic_inc_dt = build_basic_inc(legacy_dt, desc_suffix="（旧口径）")
    org_inc_dt = build_org_inc(legacy_dt, desc_suffix="（旧口径）")
    basic_yoy_dt = build_basic_yoy(legacy_dt, desc_suffix="（旧口径）")
    org_yoy_dt = build_org_yoy(legacy_dt, desc_suffix="（旧口径）")

    log_stage("构建含当月 rec_n（供第1/2/3/4/6/9层使用）...", show_progress)
    current_base = build_rec_base(date_current, include_current=True, desc_suffix="（含当月）")
    current_avg_prop = build_avg_prop(current_base, date_current, desc_suffix="（含当月）")
    current_partial = pd.concat([current_base, current_avg_prop], axis=1)
    current_org = build_org_features(current_partial, date_current, desc_suffix="（含当月）")
    current_dt = pd.concat([current_partial, current_org], axis=1)

    compare_dt = build_time_compare(current_dt, [1, 3, 6, 12, 24], desc_suffix="（含当月）")
    diff_dt = build_diff_metrics(current_dt, date_current, desc_suffix="（含当月）")
    bcto_dt = build_bcto(current_dt, [0, 1, 3, 6, 12, 18, 24], desc_suffix="（含当月）")
    month_activity_dt = build_month_activity_features(desc_suffix="（含当月）")
    fail_out_month_dt = build_fail_out_month_features(desc_suffix="（含当月）")
    time_dt = build_time_features()

    dt = pd.concat(
        [
            current_dt,
            compare_dt,
            diff_dt,
            month_activity_dt,
            fail_out_month_dt,
            time_dt,
            basic_inc_dt,
            org_inc_dt,
            basic_yoy_dt,
            org_yoy_dt,
            bcto_dt,
        ],
        axis=1,
    )

    # 先将 dict 型字段转为数值，再统一应用窗口 missing / 补0 规则。
    object_cols = [col for col in dt.columns if dt[col].dtype == "O"]
    for col in progress_iter(object_cols, desc="将 dict 型字段转数值", total=len(object_cols), enabled=show_progress):
        dt[col] = [dict_max(value) for value in dt[col]]

    for month in [0, 1, 3, 6, 12, 18, 24]:
        for org in org_type3:
            dt[f"r{month}_{org}_sf_out_mad"] = dict_max_series(current_dt[f"rec_{month}_{org}_suc_out_max"]) - dict_max_series(current_dt[f"rec_{month}_{org}_fail_out_max"])

    dt = apply_window_activity_rules(dt)
    time_dt = apply_time_activity_rules(time_dt)
    dt[time_dt.columns] = time_dt

    # 生成完整 V6 变量后，V7 输出阶段只保留目标字典中的非 verify 变量。
    dt_all_res = pd.concat(
        [data[["index"] + key_cols + ["cardnum"]].reset_index(drop=True), dt.reset_index(drop=True)],
        axis=1,
    )
    dt_all_res["fee_status"] = dt_all_res["cardnum"].fillna(0).gt(0).astype(int)

    selected_vars = resolve_target_vars()
    if selected_vars is not None:
        base_cols = [col for col in ["index"] + key_cols + ["cardnum", "fee_status"] if col in dt_all_res.columns]
        target_cols = [col for col in selected_vars if col not in base_cols]
        available_cols = [col for col in target_cols if col in dt_all_res.columns]
        missing_cols = [col for col in target_cols if col not in dt_all_res.columns]
        if missing_cols:
            msg = f"目标变量未生成（已忽略 verify* 变量）: {missing_cols[:20]}"
            if len(missing_cols) > 20:
                msg += f" ... 共 {len(missing_cols)} 个"
            if strict_target_vars:
                raise KeyError(msg)
            warnings.warn(msg)
        dt_all_res = dt_all_res[base_cols + available_cols]
        log_stage(f"按目标字典保留 {len(available_cols)} 个变量，基础列 {len(base_cols)} 个。", show_progress)

    output_file = str(file_name)
    lower_output = output_file.lower()
    if lower_output.endswith(".parquet"):
        dt_all_res.to_parquet(output_file, index=False)
    else:
        if not lower_output.endswith((".csv", ".csv.gz")):
            output_file = f"{output_file}.csv"
        dt_all_res.to_csv(output_file, index=False)
    log_stage(f"结果已保存到 {output_file}", show_progress)

    return dt_all_res

In [ ]:
try:
    log_stage("V7 主流程开始")
    if df is None:
        raise ValueError("请先填写 input_path")

    log_stage("开始执行 build_df_new")
    df_new = build_df_new(
        df,
        key_cols=key_cols,
        card_col="card",
        serialize_max=False,
        show_progress=True,
        checkpoint_dir=build_checkpoint_dir,
        resume_checkpoint=resume_build_checkpoint,
    )
    log_stage(f"build_df_new 执行完成: {len(df_new)} 行, {len(df_new.columns)} 列")

    log_stage("开始执行 get")
    result_df = get(
        df_new,
        output_name,
        show_progress=True,
        pre_aggregated=True,
        target_vars_path=target_vars_path,
        strict_target_vars=strict_target_vars,
        key_cols=key_cols,
    )
    log_stage(f"get 执行完成: {len(result_df)} 行, {len(result_df.columns)} 列")

    import gc
    gc.collect()
    log_stage("V7 主流程完成")
except Exception:
    error_text = traceback.format_exc()
    log_stage("V7 主流程失败，完整 traceback 如下：")
    write_run_log(error_text)
    print(error_text, flush=True)
    raise

result_df.head()